## Setup

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
import tensorflow_datasets as tfds
import matplotlib.pyplot as plt
import numpy as np

## Data

### Load dataset from TFDS

In [ ]:
import os
import glob
import tensorflow as tf
from PIL import Image
import numpy as np

# Download and extract the dataset
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/images.tar.gz
!wget http://www.robots.ox.ac.uk/~vgg/data/pets/data/annotations.tar.gz
!tar -xvzf images.tar.gz && tar -xvzf annotations.tar.gz
!rm images/*.mat

# Define paths
image_dir = 'images'
mask_dir = 'annotations/trimaps'

# Get file paths
image_paths = sorted(glob.glob(os.path.join(image_dir, '*.jpg')))
mask_paths = sorted(glob.glob(os.path.join(mask_dir, '*.png')))

# Split the dataset
train_size = int(0.8 * len(image_paths))
train_image_paths = image_paths[:train_size]
train_mask_paths = mask_paths[:train_size]
test_image_paths = image_paths[train_size:]
test_mask_paths = mask_paths[train_size:]

def load_image(image_path, mask_path):
    image = tf.io.read_file(image_path)
    image = tf.image.decode_jpeg(image, channels=3)
    mask = tf.io.read_file(mask_path)
    mask = tf.image.decode_png(mask, channels=1)
    return image, mask

def resize(input_image, input_mask):
    input_image = tf.image.resize(input_image, (128, 128), method="nearest")
    input_mask = tf.image.resize(input_mask, (128, 128), method="nearest")
    return input_image, input_mask

def augment(input_image, input_mask):
    if tf.random.uniform(()) > 0.5:
        input_image = tf.image.flip_left_right(input_image)
        input_mask = tf.image.flip_left_right(input_mask)
    return input_image, input_mask

def normalize(input_image, input_mask):
    input_image = tf.cast(input_image, tf.float32) / 255.0
    input_mask -= 1
    return input_image, input_mask

def load_image_train(image_path, mask_path):
    input_image, input_mask = load_image(image_path, mask_path)
    input_image, input_mask = resize(input_image, input_mask)
    input_image, input_mask = augment(input_image, input_mask)
    input_image, input_mask = normalize(input_image, input_mask)
    return input_image, input_mask

def load_image_test(image_path, mask_path):
    input_image, input_mask = load_image(image_path, mask_path)
    input_image, input_mask = resize(input_image, input_mask)
    input_image, input_mask = normalize(input_image, input_mask)
    return input_image, input_mask

# Create TensorFlow datasets
AUTOTUNE = tf.data.AUTOTUNE
BATCH_SIZE = 32

train_dataset = tf.data.Dataset.from_tensor_slices((train_image_paths, train_mask_paths))
train_dataset = train_dataset.map(load_image_train, num_parallel_calls=AUTOTUNE)
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

test_dataset = tf.data.Dataset.from_tensor_slices((test_image_paths, test_mask_paths))
test_dataset = test_dataset.map(load_image_test, num_parallel_calls=AUTOTUNE)
test_dataset = test_dataset.batch(BATCH_SIZE).prefetch(AUTOTUNE)

Take a look at the dataset info. Note the `test` and `train` data split is already built in the dataset.

### Data Visualization

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def display(display_list):
    plt.figure(figsize=(15, 15))

    title = ["Input Image", "True Mask"]

    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i+1)
        plt.title(title[i])
        if i == 0:  # For the input image
            plt.imshow(display_list[i])
        else:  # For the mask
            plt.imshow(display_list[i], cmap='gray')
        plt.axis("off")
    plt.show()

# Get a sample batch
sample_batch = next(iter(test_dataset))
random_index = np.random.choice(sample_batch[0].shape[0])
sample_image, sample_mask = sample_batch[0][random_index], sample_batch[1][random_index]

# Display the sample
display([sample_image.numpy(), sample_mask.numpy()])

## U-Net Building blocks
Create the building blocks for making the components U-Net model.

In [ ]:
def double_conv_block(x, n_filters):

    # Conv2D then ReLU activation
    x = layers.Conv2D(n_filters, 3, padding = "same", activation = "relu", kernel_initializer = "he_normal")(x)
    # Conv2D then ReLU activation
    x = layers.Conv2D(n_filters, 3, padding = "same", activation = "relu", kernel_initializer = "he_normal")(x)

    return x

In [ ]:
def downsample_block(x, n_filters):
    f = double_conv_block(x, n_filters)
    p = layers.MaxPool2D(2)(f)
    p = layers.Dropout(0.3)(p)

    return f, p

In [ ]:
def upsample_block(x, conv_features, n_filters):
    # upsample
    x = layers.Conv2DTranspose(n_filters, 3, 2, padding="same")(x)
    # concatenate
    x = layers.concatenate([x, conv_features])
    # dropout
    x = layers.Dropout(0.3)(x)
    # Conv2D twice with ReLU activation
    x = double_conv_block(x, n_filters)

    return x

## Build the U-Net Model

In [ ]:
def build_unet_model():

    # inputs
    inputs = layers.Input(shape=(128,128,3))

    # encoder: contracting path - downsample
    # 1 - downsample
    f1, p1 = downsample_block(inputs, 64)
    # 2 - downsample
    f2, p2 = downsample_block(p1, 128)
    # 3 - downsample
    f3, p3 = downsample_block(p2, 256)
    # 4 - downsample
    f4, p4 = downsample_block(p3, 512)

    # 5 - bottleneck
    bottleneck = double_conv_block(p4, 1024)

    # decoder: expanding path - upsample
    # 6 - upsample
    u6 = upsample_block(bottleneck, f4, 512)
    # 7 - upsample
    u7 = upsample_block(u6, f3, 256)
    # 8 - upsample
    u8 = upsample_block(u7, f2, 128)
    # 9 - upsample
    u9 = upsample_block(u8, f1, 64)

    # outputs
    outputs = layers.Conv2D(3, 1, padding="same", activation = "softmax")(u9)

    # unet model with Keras Functional API
    unet_model = tf.keras.Model(inputs, outputs, name="U-Net")

    return unet_model

In [ ]:
unet_model = build_unet_model()

In [ ]:
unet_model.summary()

## Compile and Train U-Net

In [ ]:
unet_model.compile(optimizer=tf.keras.optimizers.Adam(),
                   loss="sparse_categorical_crossentropy",
                   metrics=["accuracy"])

In [ ]:
import tensorflow as tf

# Define constants
NUM_EPOCHS = 5
BATCH_SIZE = 32  # Make sure this is defined

# Get the total number of samples
total_samples = len(image_paths)
train_size = int(0.8 * total_samples)
test_size = total_samples - train_size

# Calculate steps per epoch and validation steps
TRAIN_LENGTH = train_size
STEPS_PER_EPOCH = TRAIN_LENGTH // BATCH_SIZE

VAL_SUBSPLITS = 5
TEST_LENGTH = test_size
VALIDATION_STEPS = TEST_LENGTH // BATCH_SIZE // VAL_SUBSPLITS

# Assuming you've already created your model and named it unet_model
model_history = unet_model.fit(
    train_dataset,
    epochs=NUM_EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    validation_steps=VALIDATION_STEPS,
    validation_data=test_dataset
)

## Learning curve from model history

In [ ]:
def display_learning_curves(history):
    acc = history.history["accuracy"]
    val_acc = history.history["val_accuracy"]

    loss = history.history["loss"]
    val_loss = history.history["val_loss"]

    epochs_range = range(NUM_EPOCHS)

    fig = plt.figure(figsize=(12,6))

    plt.subplot(1,2,1)
    plt.plot(epochs_range, acc, label="train accuracy")
    plt.plot(epochs_range, val_acc, label="validataion accuracy")
    plt.title("Accuracy")
    plt.xlabel("Epoch")
    plt.ylabel("Accuracy")
    plt.legend(loc="lower right")

    plt.subplot(1,2,2)
    plt.plot(epochs_range, loss, label="train loss")
    plt.plot(epochs_range, val_loss, label="validataion loss")
    plt.title("Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend(loc="upper right")

    fig.tight_layout()
    plt.show()

In [ ]:
# Display learning curves
display_learning_curves(unet_model.history)

## Predictions with U-Net model
Let's try the trained U-Net model on a few samples from the test dataset.

In [ ]:
def create_mask(pred_mask):
    # No sigmoid needed if model already has it
    pred_mask = tf.where(pred_mask > 0.5, 1, 0)
    return pred_mask

In [ ]:
def show_predictions(dataset=None, num=1):
    if dataset:
        for image, mask in dataset.take(num):
            pred_mask = unet_model.predict(image)
            display([image[0], mask[0], create_mask(pred_mask)[0]])
    else:
        # Assuming sample_image is defined
        sample_pred = unet_model.predict(sample_image[tf.newaxis, ...])
        display([sample_image, sample_mask, create_mask(sample_pred)[0]])

In [ ]:
import matplotlib.pyplot as plt

def display(display_list):
    plt.figure(figsize=(15, 5 * len(display_list)))
    title = ['Input Image', 'True Mask', 'Predicted Mask']
    for i in range(len(display_list)):
        plt.subplot(1, len(display_list), i+1)
        plt.title(title[i])
        plt.imshow(tf.keras.utils.array_to_img(display_list[i]), cmap='viridis')
        plt.axis('off')
    plt.show()

In [ ]:
def show_raw_predictions(dataset=None, num=1):
    if dataset:
        for image, mask in dataset.take(num):
            pred_mask = unet_model.predict(image)
            print("Min value:", np.min(pred_mask))
            print("Max value:", np.max(pred_mask))
            print("Mean value:", np.mean(pred_mask))
            # Visualize raw predictions
            plt.figure(figsize=(10, 4))
            plt.subplot(1, 2, 1)
            plt.imshow(image[0])
            plt.title("Input Image")
            plt.subplot(1, 2, 2)
            plt.imshow(pred_mask[0, ..., 0], cmap='viridis')
            plt.colorbar()
            plt.title("Raw Predictions")
            plt.show()
    else:
        sample_pred = unet_model.predict(sample_image[tf.newaxis, ...])
        print("Min value:", np.min(sample_pred))
        print("Max value:", np.max(sample_pred))
        print("Mean value:", np.mean(sample_pred))

In [ ]:
show_raw_predictions(test_dataset)

In [ ]:
show_predictions(test_dataset.skip(5), 3)